In [4]:
import pandas as pd
import re
import os
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC


class MovieGenreClassifier:

    def __init__(self, file_path):
        self.file_path = file_path
        self.vectorizer = TfidfVectorizer(
            max_features=30000,
            ngram_range=(1, 2),
            stop_words='english'
        )
        self.models = {
            "Logistic": LogisticRegression(max_iter=1000),
            "NaiveBayes": MultinomialNB(),
            "SVM": LinearSVC()
        }
        self.best_model = None

    def load_data(self):
        dataset = []
        with open(self.file_path, encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split(" ::: ")
                if len(parts) == 4:
                    dataset.append(parts)

        df = pd.DataFrame(dataset, columns=["id", "title", "genre", "plot"])
        return df

    def clean_text(self, text):
        text = text.lower()
        text = re.sub(r'[^a-z\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    def prepare(self, df):
        df["processed"] = df["plot"].apply(self.clean_text)

        top = df["genre"].value_counts().head(10).index
        df = df[df["genre"].isin(top)]

        X = self.vectorizer.fit_transform(df["processed"])
        y = df["genre"]

        return train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    def visualize(self, df):
        counts = df["genre"].value_counts().head(10)
        plt.figure(figsize=(8, 4))
        sns.barplot(x=counts.values, y=counts.index)
        plt.title("Top 10 Genres Distribution")
        plt.savefig("genre_visual.png")
        plt.close()

    def train(self, X_train, y_train, X_test, y_test):
        results = {}
        best_score = 0

        for name, model in self.models.items():
            model.fit(X_train, y_train)
            preds = model.predict(X_test)
            acc = accuracy_score(y_test, preds)

            print(f"\n🔹 {name} Accuracy: {acc:.4f}")
            print(classification_report(y_test, preds))

            results[name] = acc

            if acc > best_score:
                best_score = acc
                self.best_model = model
                best_preds = preds
                best_true = y_test

        self.plot_results(results)
        self.plot_confusion(best_true, best_preds)

        print(f"\nFinal Selected Model: {type(self.best_model).__name__}")
        print(f"Accuracy: {best_score:.4f}")

    def plot_results(self, results):
        plt.bar(results.keys(), results.values())
        plt.ylim(0, 1)
        plt.title("Model Performance Comparison")
        plt.savefig("model_results.png")
        plt.close()

    def plot_confusion(self, y_true, y_pred):
        cm = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, cmap="coolwarm")
        plt.title("Confusion Matrix")
        plt.savefig("confusion.png")
        plt.close()

    def save_model(self):
        joblib.dump((self.best_model, self.vectorizer), "genre_classifier.pkl")
        print("Model saved successfully")

    def predict(self, text):
        cleaned = self.clean_text(text)
        vec = self.vectorizer.transform([cleaned])
        return self.best_model.predict(vec)[0]


if __name__ == "__main__":

    print("\nMovie Genre Classification System Started\n")

    FILE = "train_data.txt"

    if not os.path.exists(FILE):
        print("Dataset not found. Please check file path.")
        exit()

    system = MovieGenreClassifier(FILE)

    data = system.load_data()
    system.visualize(data)

    X_train, X_test, y_train, y_test = system.prepare(data)

    system.train(X_train, y_train, X_test, y_test)

    system.save_model()

    print("\nTry your own movie plots below!")

    while True:
        user_input = input("\nEnter plot (type 'exit' to stop): ")
        if user_input.lower() == "exit":
            print("Exiting system...")
            break

        result = system.predict(user_input)
        print(f"Predicted Genre: {result.upper()}")


Movie Genre Classification System Started


🔹 Logistic Accuracy: 0.6534
              precision    recall  f1-score   support

      action       0.68      0.25      0.36       263
      comedy       0.60      0.59      0.59      1489
 documentary       0.72      0.88      0.79      2619
       drama       0.60      0.79      0.68      2723
      family       1.00      0.06      0.12       157
      horror       0.78      0.56      0.66       441
  reality-tv       0.68      0.12      0.20       177
       short       0.60      0.30      0.40      1015
    thriller       0.57      0.10      0.17       318
     western       0.98      0.68      0.80       206

    accuracy                           0.65      9408
   macro avg       0.72      0.43      0.48      9408
weighted avg       0.66      0.65      0.62      9408


🔹 NaiveBayes Accuracy: 0.5596


C:\Users\yashw\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\yashw\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\yashw\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


              precision    recall  f1-score   support

      action       1.00      0.00      0.01       263
      comedy       0.66      0.32      0.43      1489
 documentary       0.63      0.90      0.74      2619
       drama       0.48      0.85      0.62      2723
      family       0.00      0.00      0.00       157
      horror       0.86      0.07      0.13       441
  reality-tv       0.00      0.00      0.00       177
       short       0.92      0.03      0.06      1015
    thriller       0.00      0.00      0.00       318
     western       1.00      0.15      0.26       206

    accuracy                           0.56      9408
   macro avg       0.56      0.23      0.23      9408
weighted avg       0.61      0.56      0.47      9408


🔹 SVM Accuracy: 0.6491
              precision    recall  f1-score   support

      action       0.59      0.37      0.45       263
      comedy       0.58      0.59      0.58      1489
 documentary       0.75      0.84      0.80      2619



Enter plot (type 'exit' to stop):  exit


Exiting system...
